# 10 — Fine-tuning Readiness

A prompt benchmark után jogos kérdés: mikor érdemes fine-tuningot használni? Ebben a projektben a training nem indul el automatikusan, de leakage-safe SFT adatot készítünk és ugyanazzal a final holdouttal lehet majd benchmarkolni a fine-tuned modellt.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "02_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root:", PROJECT_ROOT)

Project root: <PROJECT_ROOT>


In [2]:
dev_path=PROJECT_ROOT/"01_data/processed/development.csv"
if dev_path.exists():
    dev=pd.read_csv(dev_path)
    print(dev.shape)
    display(dev["true_label"].value_counts().sort_index().to_frame("count"))
else:
    print("Run prepare_data.py first.")

(300, 8)


,count
true_label,
api,50
billing,50
cancellation,50
complaint,50
technical,50
upgrade,50


## Leakage szabály
- few-shot examples: külön halmaz
- prompt development / SFT training: development split
- final model/prompt comparison: benchmark split

A benchmark split soha nem lehet fine-tuning adat.

In [3]:
print("Export command:")
print("python 05_scripts/12_export_finetuning_data.py")
train_path=PROJECT_ROOT/"01_data/fine_tuning/sft_train.jsonl"
valid_path=PROJECT_ROOT/"01_data/fine_tuning/sft_validation.jsonl"
print("train exists:",train_path.exists(),"validation exists:",valid_path.exists())

Export command:
python 05_scripts/12_export_finetuning_data.py
train exists: True validation exists: True


## Fair comparison design
1. base model + zero-shot
2. base model + optimized prompt
3. fine-tuned model + minimal prompt
4. fine-tuned model + optimized prompt

Mérd ugyanazt: Macro F1, per-class F1, validity, tokens, latency, cost. A fine-tuning csak akkor indokolt, ha a quality gain és az üzemeltetési előny meghaladja a training/maintenance komplexitását.